## Step 7: MLP Client Definition (unified architecture for all three datasets)

Single MLP class used for both CIFAR-10 (input_dim=512) and MNIST (input_dim=256). Architecture: Input → ReLU → Hidden(128) → Softmax(10). This isolates the defense contribution (CRTAT, blockchain) from architectural choices.

In [ ]:
# ─── v2: Unified MLP for CIFAR-10 / CIFAR-100 / MNIST ──────────────
# CNN removed — pure-NumPy convolution was the runtime bottleneck on Colab CPU,
# and the defense contribution (CRTAT) is architecture-agnostic.
# v7: n_classes is now a first-class parameter so the same client class
#     serves CIFAR-100's 100-way softmax as well as the 10-way datasets.

class FLClient:
    """Two-layer MLP: Input(D) → ReLU → Hidden(H) → Softmax(C)."""
    def __init__(self, cid, input_dim=FEATURE_DIM, hidden=HIDDEN_DIM,
                 n_classes=N_CLASSES):
        self.cid = cid
        self.D = input_dim; self.H = hidden; self.C = n_classes
        rng = np.random.default_rng(SEED + cid)
        self.W1 = rng.standard_normal((hidden, input_dim)).astype(np.float32) * np.sqrt(2/input_dim)
        self.b1 = np.zeros(hidden, dtype=np.float32)
        self.W2 = rng.standard_normal((n_classes, hidden)).astype(np.float32) * np.sqrt(2/hidden)
        self.b2 = np.zeros(n_classes, dtype=np.float32)
        self.vW1=np.zeros_like(self.W1); self.vb1=np.zeros_like(self.b1)
        self.vW2=np.zeros_like(self.W2); self.vb2=np.zeros_like(self.b2)

    def _forward(self, X):
        self.X_in = X
        self.Z1 = X @ self.W1.T + self.b1
        self.A1 = np.maximum(0, self.Z1)
        logits = self.A1 @ self.W2.T + self.b2
        e = np.exp(logits - logits.max(axis=1, keepdims=True))
        self.probs = e / e.sum(axis=1, keepdims=True)
        return self.probs

    def _backward(self, X, y, lr, momentum=0.9, l2=1e-4):
        B = len(X)
        one_hot = np.eye(self.C, dtype=np.float32)[y]
        dL  = (self.probs - one_hot) / B
        dW2 = dL.T @ self.A1 + l2 * self.W2
        db2 = dL.sum(0)
        dA1 = dL @ self.W2
        dZ1 = dA1 * (self.Z1 > 0)
        dW1 = dZ1.T @ X + l2 * self.W1
        db1 = dZ1.sum(0)
        self.vW2 = momentum*self.vW2 - lr*dW2; self.W2 += self.vW2
        self.vb2 = momentum*self.vb2 - lr*db2; self.b2 += self.vb2
        self.vW1 = momentum*self.vW1 - lr*dW1; self.W1 += self.vW1
        self.vb1 = momentum*self.vb1 - lr*db1; self.b1 += self.vb1

    def train(self, X, y, epochs=EPOCHS, lr=LR_INIT, rng=None, verbose=False):
        """v4: added optional verbose flag — prints per-epoch train loss + acc."""
        if rng is None: rng = np.random.default_rng(SEED + self.cid)
        n = len(X)
        for ep in range(epochs):
            idx = rng.permutation(n)
            ep_loss = 0.0; ep_n = 0
            for start in range(0, n, BATCH_SIZE):
                b = idx[start:start+BATCH_SIZE]
                self._forward(X[b]); self._backward(X[b], y[b], lr)
                # accumulate cross-entropy loss for reporting
                if verbose:
                    eps_ = 1e-12
                    p_correct = self.probs[np.arange(len(b)), y[b]]
                    ep_loss += float(-np.log(p_correct + eps_).sum())
                    ep_n    += len(b)
            if verbose:
                # per-epoch train accuracy on full local data
                tr_acc = (self.predict_proba(X).argmax(1) == y).mean()
                print(f'      epoch {ep+1}/{epochs}  loss={ep_loss/max(ep_n,1):.4f}  train_acc={tr_acc*100:5.1f}%')

    def get_params(self):
        return np.concatenate([self.W1.flatten(), self.b1,
                               self.W2.flatten(), self.b2])

    def set_params(self, params):
        D, H, C = self.D, self.H, self.C
        ptr = 0
        self.W1 = params[ptr:ptr+H*D].reshape(H, D).astype(np.float32); ptr += H*D
        self.b1 = params[ptr:ptr+H].astype(np.float32); ptr += H
        self.W2 = params[ptr:ptr+C*H].reshape(C, H).astype(np.float32); ptr += C*H
        self.b2 = params[ptr:ptr+C].astype(np.float32)

    def predict_proba(self, X):
        return self._forward(X)

    @property
    def param_size(self):
        return self.D*self.H + self.H + self.H*self.C + self.C


def cosine_lr(rnd, total_rounds, lr_init=LR_INIT, lr_min=LR_MIN):
    return lr_min + 0.5*(lr_init-lr_min)*(1+np.cos(np.pi*rnd/total_rounds))

def evaluate_model(params, X_test, y_test, input_dim, n_classes=N_CLASSES):
    cl = FLClient(-1, input_dim=input_dim, n_classes=n_classes)
    cl.set_params(params)
    probs = cl.predict_proba(X_test)
    preds = probs.argmax(1)
    return (preds == y_test).mean(), probs

def make_mlp_clients(n, input_dim=FEATURE_DIM, n_classes=N_CLASSES):
    return [FLClient(i, input_dim=input_dim, n_classes=n_classes) for i in range(n)]

# CIFAR-10 / MNIST use 10-way softmax; CIFAR-100 uses 100-way softmax
clients_cifar    = make_mlp_clients(NUM_CLIENTS, input_dim=CIFAR_PCA_DIM,    n_classes=N_CLASSES)
clients_cifar100 = make_mlp_clients(NUM_CLIENTS, input_dim=CIFAR100_PCA_DIM, n_classes=N_CLASSES_C100)
clients_mnist    = make_mlp_clients(NUM_CLIENTS, input_dim=MNIST_PCA_DIM,    n_classes=N_CLASSES)

CIFAR_PARAM_SIZE    = clients_cifar[0].param_size
CIFAR100_PARAM_SIZE = clients_cifar100[0].param_size
MNIST_PARAM_SIZE    = clients_mnist[0].param_size

print('✅ MLP clients instantiated for all three datasets.')
print(f'   CIFAR-10  MLP : {CIFAR_PCA_DIM} → ReLU → {HIDDEN_DIM} → Softmax(10)   '
      f'({CIFAR_PARAM_SIZE:,} params)')
print(f'   CIFAR-100 MLP : {CIFAR100_PCA_DIM} → ReLU → {HIDDEN_DIM} → Softmax(100)  '
      f'({CIFAR100_PARAM_SIZE:,} params)')
print(f'   MNIST     MLP : {MNIST_PCA_DIM} → ReLU → {HIDDEN_DIM} → Softmax(10)   '
      f'({MNIST_PARAM_SIZE:,} params)')
print(f'   Optimiser     : SGD + momentum(0.9) + L2(1e-4) + cosine LR (init={LR_INIT})')


✅ MLP clients instantiated for all three datasets.
   CIFAR-10  MLP : 512 → ReLU → 128 → Softmax(10)   (66,954 params)
   CIFAR-100 MLP : 512 → ReLU → 128 → Softmax(100)  (78,564 params)
   MNIST     MLP : 256 → ReLU → 128 → Softmax(10)   (34,186 params)
   Optimiser     : SGD + momentum(0.9) + L2(1e-4) + cosine LR (init=0.08)
